# 4. Feature Extraction
Bu notebook, temizlenmiş yorum verilerinden makine öğrenmesi ve derin öğrenme modelleri için özellik vektörleri (TF-IDF ve Sequence) oluşturur.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import scipy.sparse
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("Veri yükleniyor...")
df = pd.read_csv('data/reviews_preprocessed.csv')
# Null değerleri temizle (önceki adımlarda kalmış olabilir)
df = df.dropna(subset=['cleaned_text'])
print(f"Veri yüklendi, Shape: {df.shape}")


## 4.1 TF-IDF (Klasik Modeller için)

In [ ]:
print("TF-IDF dönüşümü uygulanıyor...")
tfidf_vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.95
)

X_tfidf = tfidf_vectorizer.fit_transform(df['cleaned_text'])
print(f"TF-IDF Shape: {X_tfidf.shape}")

print("\nEn Sık 20 TF-IDF Özelliği:")
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
sum_tfidf = X_tfidf.sum(axis=0).A1
top_indices = sum_tfidf.argsort()[-20:][::-1]
for i, idx in enumerate(top_indices, 1):
    print(f"{i}. {feature_names[idx]} ({sum_tfidf[idx]:.2f})")


In [ ]:
print("Sayısal ekstra özellikler normalize ediliyor...")
numeric_features = [
    'review_length', 'word_count', 'exclamation_count', 'question_count',
    'avg_word_length', 'uppercase_ratio', 'sentiment_polarity', 'sentiment_subjectivity'
]

scaler = StandardScaler()
X_numeric = scaler.fit_transform(df[numeric_features])

print("TF-IDF ve Sayısal özellikler birleştiriliyor (hstack)...")
X_combined = scipy.sparse.hstack([X_tfidf, X_numeric]).tocsr()
print(f"Final Birleştirilmiş Shape: {X_combined.shape}")


In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(tfidf_vectorizer, 'models/tfidf_vectorizer.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

# TF-IDF matrisini diske kaydetmek devasa olabilir, şimdilik sadece indexleri veya 
# istendiği gibi modelleri kaydediyoruz.
print("✅ TF-IDF kaydedildi")


## 4.2 Sequence Encoding (Derin Öğrenme için)

In [ ]:
print("Keras Tokenizer eğitiliyor...")
tokenizer = Tokenizer(num_words=50000, oov_token="<OOV>")
tokenizer.fit_on_texts(df['cleaned_text'])

vocab_size = len(tokenizer.word_index) + 1
print(f"Vocab Size (Toplam Eşsiz Kelime): {vocab_size}")


In [ ]:
print("Metinler dizilere (sequence) dönüştürülüyor...")
sequences = tokenizer.texts_to_sequences(df['cleaned_text'])

print("Padding işlemi uygulanıyor...")
X_seq = pad_sequences(sequences, maxlen=200, padding='post', truncating='post')
print(f"Sequence Shape: {X_seq.shape}")

print("\nÖrnek 3 Sequence:")
for i in range(3):
    print(f"Örnek {i+1}: {X_seq[i][:15]}... (uzunluk: {len(X_seq[i])})")


In [ ]:
os.makedirs('features', exist_ok=True)
joblib.dump(tokenizer, 'models/tokenizer.pkl')
np.save('features/sequences.npy', X_seq)
print("✅ Sequence kaydedildi")


## 4.3 Train / Validation / Test Split

In [ ]:
print("Veri seti bölünüyor (%70 Train, %15 Val, %15 Test)...")
indices = np.arange(len(df))
y = df['label'].values

# İlk önce Train ve Kalan (Val+Test) olarak ayır: %70, %30
idx_train, idx_temp, y_train, y_temp = train_test_split(
    indices, y, test_size=0.30, random_state=42, stratify=y
)

# Kalanı Val ve Test olarak ikiye böl: %50, %50 (Her biri genel toplamın %15'i olur)
idx_val, idx_test, y_val, y_test = train_test_split(
    idx_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("\n--- Veri Boyutları ---")
print(f"Eğitim (Train)     : {len(idx_train)}")
print(f"Doğrulama (Val)    : {len(idx_val)}")
print(f"Test (Test)        : {len(idx_test)}")

print("\n--- Sınıf Dağılımları (Stratified Doğrulaması) ---")
def print_dist(y_subset, name):
    unique, counts = np.unique(y_subset, return_counts=True)
    dist = {u: f"%{c/len(y_subset)*100:.1f}" for u, c in zip(unique, counts)}
    print(f"{name:10} Sınıf Dağılımı: {dist}")

print_dist(y_train, "Train")
print_dist(y_val, "Val")
print_dist(y_test, "Test")


In [ ]:
np.save('features/train_idx.npy', idx_train)
np.save('features/val_idx.npy', idx_val)
np.save('features/test_idx.npy', idx_test)
np.save('features/labels.npy', y)
print("✅ Split kaydedildi")

summary_df = pd.DataFrame({
    'Set': ['Train', 'Validation', 'Test'],
    'Örnek Sayısı': [len(idx_train), len(idx_val), len(idx_test)],
    'Oran (%)': [len(idx_train)/len(y)*100, len(idx_val)/len(y)*100, len(idx_test)/len(y)*100]
})
display(summary_df)
